In [8]:
!pip install tabpfn-client

  Using cached anyio-4.9.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Preparing metadata (setup.py) ... done
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.4 MB/s eta 0:00:00
Using cached anyio-4.9.0-py3-none-any.whl (100 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144555 sha256=685b57a60f358fc729719b91bcaaacbe69fc5f7ce3af8747ca3b6f869d1cb56d
  Stored in directory: /Users/mduranfrigola/Library/Caches/pip/wheels/1f/be/48/13754633f1d08d1fbfc60d5e80ae1e5d7329500477685286cd
Successfully built antlr4-python3-runtime


In [9]:
from tabpfn_client import TabPFNClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from tqdm import tqdm
import json

import pandas as pd
from sklearn.impute import SimpleImputer


In [10]:
def binarize(s):
    print(s.value_counts())
    counts = s.value_counts()
    minority_class = counts.idxmin()
    majority_class = counts.idxmax()
    binary_series = s.map({minority_class: 1, majority_class: 0})
    return binary_series


def split_data(X, y, test_size, random=False):
    if random:
        return train_test_split(X, y, stratify=y, test_size=test_size)
    else:

        pos_idxs = []
        for idx, y_ in enumerate(list(y)):
            if y_ == 1:
                pos_idxs += [idx]

        n = int(len(pos_idxs)*(1-test_size))
        cut_idx = pos_idxs[n]
                
        # Test set: only rows with indices in `test_pos_indices`
        X_test = X.iloc[cut_idx:]
        y_test = y.iloc[cut_idx:]

        # Train set: all other rows
        X_train = X.iloc[:cut_idx]
        y_train = y.iloc[:cut_idx]
    
        return X_train, X_test, y_train, y_test


def cv_tabpfn(df, test_size=0.33, random=True, impute=False):
    columns = list(df.columns)
    df = df[df[columns[0]].notnull()]
    y = binarize(df[columns[0]])
    x = df[columns[1:]]
    x_train, x_test, y_train, y_test = split_data(x, y, test_size, random)
    if impute:
        imputer = SimpleImputer()
        imputer.fit(x_train)
        x_train = imputer.transform(x_train)
        x_test = imputer.transform(x_test)
    print("TabPFN Classifier")
    mdl = TabPFNClassifier(ignore_pretraining_limits=True)
    mdl.fit(x_train, y_train)
    print("Training done")
    y_hat = mdl.predict_proba(x_test)[:,1]
    print("Predicting done")
    fpr, tpr, _ = roc_curve(y_test, y_hat)
    data = {"fpr": [float(x) for x in fpr], "tpr": [float(x) for x in tpr], "auroc": auc(fpr, tpr)}
    print(data["auroc"])
    return data


dataset_names = ["ml_d1_predelivery", "ml_d2_earlydeath", "ml_d3_latedeath"]

for dn in dataset_names:
    df = pd.read_csv("../data/processed/{0}.csv".format(dn))
    cv_data = []
    for _ in tqdm(range(10)):
        cv_data += [cv_tabpfn(df, random=True)]
    with open("../results/cv_{0}.json".format(dn), "w") as f:
        json.dump(cv_data, f, indent=4)

  0%|          | 0/10 [00:00<?, ?it/s]

Outcome Death
Live birth                   10422
Miscarriage or Stillbirth      174
Name: count, dtype: int64
TabPFN Classifier
  Welcome to TabPFN!

  TabPFN is still under active development, and we are working hard to make it better.
  Please bear with us if you encounter any issues.


Opening browser for login. Please complete the login/registration process in your browser and return here.

  Login via browser successful!
Training done


Processing: 100%|██████████| [00:12<00:00]
 10%|█         | 1/10 [02:20<21:02, 140.30s/it]

Predicting done
0.7912994696042431
Outcome Death
Live birth                   10422
Miscarriage or Stillbirth      174
Name: count, dtype: int64
TabPFN Classifier
Training done


Processing: 100%|██████████| [00:15<00:00]
 20%|██        | 2/10 [02:38<09:10, 68.76s/it] 

...

Predicting done
0.807688188494492


  0%|          | 0/10 [00:00<?, ?it/s]

Early Neonatal Death
Alive    10297
Dead       112
Name: count, dtype: int64
TabPFN Classifier
Training done


  0%|          | 0/10 [00:03<?, ?it/s]


RuntimeError: Fail to call predict with error: {'detail': "API Daily Limit Reached. Your account's current limit is 5000000 table cell predictions per day. This limit resets daily at 00:00:00 UTC. You can make your next API request after 2025-05-28 00:00:00 UTC. To learn more about usage limits or request an increase, please visit: https://priorlabs.ai/api-usage-limit/"}

Early Neonatal Death
Alive    10297
Dead       112
Name: count, dtype: int64

In [15]:
mdl = TabPFNClassifier(ignore_pretraining_limits=True)
mdl.fit(df, df["Pre-term Delivery"])

/home/mduranfrigola/miniconda3/envs/tabpfn/lib/python3.12/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 10422 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(


TabPFNClassifier(ignore_pretraining_limits=True)

In [11]:
df

,Pre-term Delivery,Miscarriage,Outcome Death,Maternal Age,School Level,Years of Education,Parity,Maternal Height,Maternal Weight,Multiple Birth
0,Term,No miscarriage,Live birth,28.0,3=School,7.0,2.0,167.0,87.0,0.0
1,Term,No miscarriage,Live birth,19.0,3=School,7.0,0.0,154.0,66.0,0.0
2,Term,No miscarriage,Live birth,32.0,3=School,7.0,2.0,NaN,67.0,0.0
3,Term,No miscarriage,Live birth,32.0,3=School,9.0,3.0,154.0,97.0,0.0
4,Term,No miscarriage,Live birth,17.0,3=School,6.0,1.0,153.0,71.0,0.0
...,...,...,...,...,...,...,...,...,...,...
10591,Term,No miscarriage,Live birth,22.0,3=School,11.0,0.0,166.0,NaN,0.0
10592,Preterm,No miscarriage,Live birth,22.0,3=School,10.0,0.0,154.0,NaN,0.0
10593,Preterm,No miscarriage,Live birth,36.0,3=School,9.0,3.0,154.0,NaN,0.0
10594,Preterm,No miscarriage,Live birth,19.0,3=School,7.0,0.0,NaN,55.0,0.0
